<a href="https://colab.research.google.com/github/Tipusultan199/kkk/blob/main/Phase1A_Colab_Cleaning_Debug_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1A Metadata Cleaning Debug Notebook

This notebook reorganizes the original Phase 1A cleaning logic into small Colab cells.  
Each cell prints what changed and reports either **✅ CLEANING APPLIED CORRECTLY** or **❌ CLEANING NOT FULLY APPLIED** for that step.

In [10]:
# ============================================================
# CELL 1 — Imports and Colab path setup
# This cell imports libraries and sets the input/output paths.
# ============================================================

import re
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

# Change this only if your uploaded file has a different name.
INPUT_CSV = Path("/content/063_T24.csv")

# Cleaned output will be saved here.
OUTPUT_DIR = Path("/content/phase1A_cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If subject_id cannot be parsed from filename/folder, set it manually, e.g., 63.
SUBJECT_ID_OVERRIDE = 10

print("INPUT_CSV :", INPUT_CSV)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("IN_COLAB  :", IN_COLAB)

print("\n✅ CLEANING APPLIED CORRECTLY: Setup cell loaded.")

INPUT_CSV : /content/063_T24.csv
OUTPUT_DIR: /content/phase1A_cleaned
IN_COLAB  : True

✅ CLEANING APPLIED CORRECTLY: Setup cell loaded.


In [11]:
# ============================================================
# CELL 2 — Upload CSV if it is not already in /content
# This cell helps you upload the raw iMotions CSV in Colab.
# ============================================================

if not INPUT_CSV.exists():
    print(f"File not found at: {INPUT_CSV}")
    if IN_COLAB:
        print("Please upload your raw CSV file now...")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("No file was uploaded.")
        first_uploaded_name = list(uploaded.keys())[0]
        INPUT_CSV = Path("/content") / first_uploaded_name
        print("Uploaded file:", INPUT_CSV)
    else:
        raise FileNotFoundError(f"Please place the CSV at: {INPUT_CSV}")
else:
    print("CSV file already found:", INPUT_CSV)

print("File exists:", INPUT_CSV.exists())
print("File size MB:", round(INPUT_CSV.stat().st_size / (1024 * 1024), 3))

if INPUT_CSV.exists() and INPUT_CSV.suffix.lower() == ".csv":
    print("✅ CLEANING APPLIED CORRECTLY: Raw CSV is ready.")
else:
    print("❌ CLEANING NOT FULLY APPLIED: CSV file is missing or not a .csv file.")

CSV file already found: /content/063_T24.csv
File exists: True
File size MB: 2.968
✅ CLEANING APPLIED CORRECTLY: Raw CSV is ready.


In [12]:
# ============================================================
# CELL 3 — Original Phase 1A settings
# This cell keeps the main cleaning policy from your original code.
# ============================================================

# Accept endings like: _T14.csv, _T106.csv, 063_T24.csv
FNAME_ANY_RE = re.compile(r"[_\-]T(?P<num>\d{2,3})\.csv$", re.IGNORECASE)
FNAME_SUBJECT_RE = re.compile(r"^(?P<sid>\d{1,4})[_\-]T\d{2,3}\.csv$", re.IGNORECASE)

# Neon world-camera FOV (px) per metadata
ET_FOV_W = 1600.0
ET_FOV_H = 1200.0

# ---- EMG policy for REAL trials ----
KEEP_EMG_RAW = True        # keep raw for your pipeline
DROP_EMG_VENDOR = True     # drop vendor mV to avoid opaque preprocessing

# Add a master seconds clock alongside originals
ADD_TIMESTAMP_SECONDS = True

# Keep distances/pupils in mm (True) or convert both to meters (False)
KEEP_DIST_IN_MM = True

# Drop eyelid features unless you plan to use them
DROP_EYELID_FEATURES = True

# Important correction for this sample:
# Some iMotions files encode slide rows as numeric EventSource=1 but still have SlideEvent values.
# This keeps your original intention: remove slide/media event rows before saving sensor data.
REMOVE_SLIDE_EVENT_ROWS_STRONG = True

# Columns to drop (meta/ops/noisy). We keep SampleNumber & time columns for synchronization/use.
DROP_COLS_EXACT_BASE = {
    "EventSource", "SlideEvent", "StimType", "Duration",
    "CollectionPhase", "SourceStimuliName",
    "Auto 1 active", "Auto 1 instance",
    "Active active", "active active", "active instance",
    # Camera/capture eye positions often useless/noisy for Neon exports
    "ET_CameraLeftX", "ET_CameraLeftY", "ET_CameraRightX", "ET_CameraRightY",
}

# Raw EMG column names
EMG_RAW_COLS = {"Ch1 EMG raw", "Ch2 EMG raw", "Ch3 EMG raw", "Ch4 EMG raw"}

# Vendor EMG (mV) columns
EMG_VENDOR_COLS = {"Ch1 EMG", "Ch2 EMG", "Ch3 EMG", "Ch4 EMG"}

# Possible extras to prune
DROP_COLS_MAYBE = {"EventSource.1", "EventSource.2", "EventSource.3"}
DROP_PREFIXES = ("Unnamed:",)

print("KEEP_EMG_RAW                 =", KEEP_EMG_RAW)
print("DROP_EMG_VENDOR              =", DROP_EMG_VENDOR)
print("ADD_TIMESTAMP_SECONDS        =", ADD_TIMESTAMP_SECONDS)
print("KEEP_DIST_IN_MM              =", KEEP_DIST_IN_MM)
print("DROP_EYELID_FEATURES         =", DROP_EYELID_FEATURES)
print("REMOVE_SLIDE_EVENT_ROWS_STRONG =", REMOVE_SLIDE_EVENT_ROWS_STRONG)

print("\n✅ CLEANING APPLIED CORRECTLY: Phase 1A settings loaded.")

KEEP_EMG_RAW                 = True
DROP_EMG_VENDOR              = True
ADD_TIMESTAMP_SECONDS        = True
KEEP_DIST_IN_MM              = True
DROP_EYELID_FEATURES         = True
REMOVE_SLIDE_EVENT_ROWS_STRONG = True

✅ CLEANING APPLIED CORRECTLY: Phase 1A settings loaded.


In [13]:
# ============================================================
# CELL 4 — Helper functions
# This cell defines parsing, timestamp, conversion, and validation helpers.
# ============================================================

def print_step_result(step_name: str, passed: bool, details: str = ""):
    print("\n" + "=" * 80)
    print(step_name)
    print("=" * 80)
    if details:
        print(details)
    if passed:
        print("✅ CLEANING APPLIED CORRECTLY")
    else:
        print("❌ CLEANING NOT FULLY APPLIED")


def find_header_row_index(path: Path) -> int:
    """Find the 'Row,' header line after iMotions metadata."""
    with path.open("r", encoding="utf-8-sig", errors="ignore") as f:
        for i, line in enumerate(f):
            if line.lstrip().startswith("Row,"):
                return i
    return 30


def parse_task_trial(name: str):
    """
    118_T14.csv  -> task=1, trial=4
    062_T106.csv -> task=10, trial=6
    063_T24.csv  -> task=2, trial=4
    """
    m = FNAME_ANY_RE.search(name)
    if not m:
        return None, None
    digits = m.group("num")
    if len(digits) < 2:
        return None, None
    trial = int(digits[-1])
    task = int(digits[:-1])
    return task, trial


def parse_subject_id_from_name(name: str):
    """Parse subject ID from Sub-63 folder name or 063_T24.csv file name."""
    m = re.search(r"Sub[\s\-_]*(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = FNAME_SUBJECT_RE.search(name)
    if m:
        return int(m.group("sid"))
    return None


def _coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _emg_raw_center_inplace(df: pd.DataFrame) -> list:
    """
    Robust-center EMG raw ADC-like counts by subtracting the median
    only if the median looks like ADC counts.
    """
    log = []
    for col in sorted(EMG_RAW_COLS):
        if col not in df.columns:
            continue
        s = _coerce_numeric(df[col])
        if not s.notna().any():
            log.append((col, "skipped: all values are NaN"))
            continue

        med = float(s.median())
        if 1e4 <= med <= 6e4:
            df[col] = (s - med).astype("float32")
            log.append((col, f"centered by median {med:.2f}; dtype=float32"))
        else:
            df[col] = s.astype("float32")
            log.append((col, f"left as-is; median {med:.2f} not in ADC range; dtype=float32"))
    return log


def apply_unit_conversions_inplace(df: pd.DataFrame, keep_dist_in_mm: bool = True) -> list:
    """
    Convert IN-PLACE without renaming columns:
    - Gaze 2D px → [0,1]
    - Distances/pupils kept in mm or converted to m
    - Validity/blink/fixation/worn → {0,1}
    - IMU/head/3D features → float32
    - EMG raw robust centering → float32
    - EEG Ch1..Ch8 → float32
    """
    log = []

    # Gaze 2D px -> [0,1], clip, float32
    for col, denom in [
        ("ET_GazeLeftx", ET_FOV_W), ("ET_GazeRightx", ET_FOV_W),
        ("ET_GazeLefty", ET_FOV_H), ("ET_GazeRighty", ET_FOV_H),
    ]:
        if col in df.columns:
            s = _coerce_numeric(df[col])
            s = (s / denom).clip(lower=0.0, upper=1.0)
            df[col] = s.astype("float32")
            log.append((col, f"px→[0,1] by {denom}; clipped; dtype=float32"))

    # Distances / pupils units
    if not keep_dist_in_mm:
        for col in ["ET_DistanceLeft", "ET_DistanceRight", "ET_PupilLeft", "ET_PupilRight"]:
            if col in df.columns:
                s = _coerce_numeric(df[col])
                mask = s >= 0
                s_conv = s.astype("float64")
                s_conv.loc[mask] = s.loc[mask] / 1000.0
                df[col] = s_conv.astype("float32")
                log.append((col, "mm→m for values >=0; negatives unchanged; dtype=float32"))
    else:
        for col in ["ET_DistanceLeft", "ET_DistanceRight", "ET_PupilLeft", "ET_PupilRight"]:
            if col in df.columns:
                df[col] = _coerce_numeric(df[col]).astype("float32")
                log.append((col, "kept in mm; dtype=float32"))

    # Validity/Blink/Fixation/Worn -> uint8
    for col in ["ET_ValidityLeftEye", "ET_ValidityRightEye", "ET_Blink", "ET_Fixation", "ET_Worn"]:
        if col in df.columns:
            s = _coerce_numeric(df[col]).fillna(0.0)
            df[col] = (s > 0.5).astype("uint8")
            log.append((col, "thresholded >0.5 → {0,1}; dtype=uint8"))

    # IMU / head rotations to float32
    for col in [
        "ET_GyroX", "ET_GyroY", "ET_GyroZ",
        "ET_AccX", "ET_AccY", "ET_AccZ",
        "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
    ]:
        if col in df.columns:
            df[col] = _coerce_numeric(df[col]).astype("float32")
            log.append((col, "dtype=float32"))

    # 3D eyeball centers / optical axis / eyelids to float32
    three_d_cols = [
        "ET_Gaze3dEyeballXLeft", "ET_Gaze3dEyeballYLeft", "ET_Gaze3dEyeballZLeft",
        "ET_Gaze3dEyeballXRight", "ET_Gaze3dEyeballYRight", "ET_Gaze3dEyeballZRight",
        "ET_Gaze3dOpticalAxisXLeft", "ET_Gaze3dOpticalAxisYLeft", "ET_Gaze3dOpticalAxisZLeft",
        "ET_Gaze3dOpticalAxisXRight", "ET_Gaze3dOpticalAxisYRight", "ET_Gaze3dOpticalAxisZRight",
        "ET_Gaze3dEyelidAngleTopLeft", "ET_Gaze3dEyelidAngleBottomLeft",
        "ET_Gaze3dEyelidAngleTopRight", "ET_Gaze3dEyelidAngleBottomRight",
        "ET_Gaze3dEyelidApertureLeft", "ET_Gaze3dEyelidApertureRight",
    ]
    for col in three_d_cols:
        if col in df.columns:
            df[col] = _coerce_numeric(df[col]).astype("float32")
            log.append((col, "dtype=float32"))

    # EMG raw robust centering
    log += _emg_raw_center_inplace(df)

    # EEG μV to float32, no scaling
    for ch in ["Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8"]:
        if ch in df.columns:
            df[ch] = _coerce_numeric(df[ch]).astype("float32")
            log.append((ch, "EEG kept in original units; dtype=float32"))

    return log


def _make_timestamp_seconds(df: pd.DataFrame):
    """
    Build a master seconds clock.
    Priority: iMotions Timestamp ms → ET_TimeSignal ms → LSL Timestamp s.
    """
    candidates = []
    if "Timestamp" in df.columns:
        candidates.append(("Timestamp", 1000.0))
    if "ET_TimeSignal" in df.columns:
        candidates.append(("ET_TimeSignal", 1000.0))
    if "LSL Timestamp" in df.columns:
        candidates.append(("LSL Timestamp", 1.0))

    for col, denom in candidates:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().any():
            t = s / denom
            t0 = t.dropna().iloc[0]
            t = (t - t0).ffill()

            dt = t.diff().dropna()
            if not dt.empty:
                positive_dt = dt[dt > 0]
                med_dt = float(positive_dt.median()) if len(positive_dt) else float(dt.median())
                est_fs = 1.0 / med_dt if med_dt > 0 else np.nan
                print(f"[INFO] {col} → Timestamp_seconds")
                print(f"       median positive Δt ≈ {med_dt:.9f} s; rough row-rate ≈ {est_fs:.2f} Hz")
                print("       Note: this is a row-level clock for interleaved streams, not native EEG/EMG/ET fs.")
            return t.astype("float64"), col

    return None, None


print_step_result("CELL 4 — Helper functions loaded", True)


CELL 4 — Helper functions loaded
✅ CLEANING APPLIED CORRECTLY


In [14]:
# ============================================================
# CELL 5 — Read raw CSV after iMotions metadata
# This cell finds the Row header and loads the actual data table.
# ============================================================

header_idx = find_header_row_index(INPUT_CSV)

df_raw = pd.read_csv(
    INPUT_CSV,
    skiprows=header_idx,
    header=0,
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="skip",
)

df_raw.columns = [c.strip() for c in df_raw.columns]

print("Header row index:", header_idx)
print("Raw shape:", df_raw.shape)
print("Number of columns:", len(df_raw.columns))
print("\nRaw columns:")
print(df_raw.columns.tolist())

print("\nRaw preview:")
display(df_raw.head(5))

passed = (header_idx >= 0) and (df_raw.shape[0] > 0) and ("Row" in df_raw.columns)
details = f"Loaded {df_raw.shape[0]} rows and {df_raw.shape[1]} columns."
print_step_result("CELL 5 — Raw CSV loaded", passed, details)

Header row index: 29
Raw shape: (11906, 74)
Number of columns: 74

Raw columns:
['Row', 'Timestamp', 'EventSource', 'SlideEvent', 'StimType', 'Duration', 'CollectionPhase', 'SourceStimuliName', 'EventSource.1', 'SampleNumber', 'Ch1 EMG raw', 'Ch1 EMG', 'Ch2 EMG raw', 'Ch2 EMG', 'Ch3 EMG raw', 'Ch3 EMG', 'Ch4 EMG raw', 'Ch4 EMG', 'EventSource.2', 'ET_TimeSignal', 'ET_PupilLeft', 'ET_PupilRight', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_CameraLeftX', 'ET_CameraLeftY', 'ET_CameraRightX', 'ET_CameraRightY', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll', 'ET_Blink', 'ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOptical

,Row,Timestamp,EventSource,SlideEvent,StimType,Duration,CollectionPhase,SourceStimuliName,EventSource.1,SampleNumber,Ch1 EMG raw,Ch1 EMG,Ch2 EMG raw,Ch2 EMG,Ch3 EMG raw,Ch3 EMG,Ch4 EMG raw,Ch4 EMG,EventSource.2,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_CameraLeftX,ET_CameraLeftY,ET_CameraRightX,ET_CameraRightY,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Gaze3dEyelidAngleTopLeft,ET_Gaze3dEyelidAngleBottomLeft,ET_Gaze3dEyelidAngleTopRight,ET_Gaze3dEyelidAngleBottomRight,ET_Gaze3dEyelidApertureLeft,ET_Gaze3dEyelidApertureRight,ET_Fixation,ET_Worn,EventSource.3,LSL Timestamp,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8
0,1,0.00000,1.0,StartSlide,TestImage,8400.0,StimuliDisplay,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,0.00000,1.0,StartMedia,TestImage,8400.0,StimuliDisplay,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,8.42220,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.676423,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
3,4,8.42235,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.676424,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
4,5,10.42220,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.678423,104432.734375,97476.398438,104738.296875,117938.625000,134648.578125,61575.230469,62170.226562,45518.308594



CELL 5 — Raw CSV loaded
Loaded 11906 rows and 74 columns.
✅ CLEANING APPLIED CORRECTLY


In [15]:
# ============================================================
# CELL 6 — Insert subject_id, task, and trial metadata
# This cell parses metadata from filename and inserts columns at the front.
# ============================================================

df = df_raw.copy()

subject_id = SUBJECT_ID_OVERRIDE
if subject_id is None:
    subject_id = parse_subject_id_from_name(INPUT_CSV.name)
if subject_id is None:
    subject_id = parse_subject_id_from_name(INPUT_CSV.parent.name)

task, trial = parse_task_trial(INPUT_CSV.name)

print("Parsed subject_id:", subject_id)
print("Parsed task      :", task)
print("Parsed trial     :", trial)

insert_cols = []
if subject_id is not None:
    insert_cols.append(("subject_id", subject_id))
if task is not None:
    insert_cols.append(("task", task))
if trial is not None:
    insert_cols.append(("trial", trial))

before_cols = list(df.columns)

for col, val in reversed(insert_cols):
    if col not in df.columns:
        df.insert(0, col, val)
    else:
        df[col] = val

after_cols = list(df.columns)

print("\nColumns inserted:")
print([c for c, _ in insert_cols])
print("Shape before metadata:", df_raw.shape)
print("Shape after metadata :", df.shape)
print("\nPreview after metadata insertion:")
display(df.head(5))

passed = all(col in df.columns for col, _ in insert_cols) and len(insert_cols) >= 3
if not passed:
    print("WARNING: subject/task/trial were not all parsed. Set SUBJECT_ID_OVERRIDE if subject_id is missing.")

print_step_result("CELL 6 — Metadata inserted", passed)

Parsed subject_id: 10
Parsed task      : 2
Parsed trial     : 4

Columns inserted:
['subject_id', 'task', 'trial']
Shape before metadata: (11906, 74)
Shape after metadata : (11906, 77)

Preview after metadata insertion:


,subject_id,task,trial,Row,Timestamp,EventSource,SlideEvent,StimType,Duration,CollectionPhase,SourceStimuliName,EventSource.1,SampleNumber,Ch1 EMG raw,Ch1 EMG,Ch2 EMG raw,Ch2 EMG,Ch3 EMG raw,Ch3 EMG,Ch4 EMG raw,Ch4 EMG,EventSource.2,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_CameraLeftX,ET_CameraLeftY,ET_CameraRightX,ET_CameraRightY,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Gaze3dEyelidAngleTopLeft,ET_Gaze3dEyelidAngleBottomLeft,ET_Gaze3dEyelidAngleTopRight,ET_Gaze3dEyelidAngleBottomRight,ET_Gaze3dEyelidApertureLeft,ET_Gaze3dEyelidApertureRight,ET_Fixation,ET_Worn,EventSource.3,LSL Timestamp,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8
0,10,2,4,1,0.00000,1.0,StartSlide,TestImage,8400.0,StimuliDisplay,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,2,4,2,0.00000,1.0,StartMedia,TestImage,8400.0,StimuliDisplay,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,2,4,3,8.42220,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.676423,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
3,10,2,4,4,8.42235,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.676424,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
4,10,2,4,5,10.42220,NaN,NaN,NaN,NaN,NaN,Neon Glasses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,535400.678423,104432.734375,97476.398438,104738.296875,117938.625000,134648.578125,61575.230469,62170.226562,45518.308594



CELL 6 — Metadata inserted
✅ CLEANING APPLIED CORRECTLY


In [16]:
# ============================================================
# CELL 7 — Remove SlideEvents / media event rows
# This cell removes non-sensor rows such as StartSlide, StartMedia, EndMedia, EndSlide.
# ============================================================

rows_before = len(df)

slide_mask = pd.Series(False, index=df.index)

# Original logic: remove rows where EventSource explicitly says SlideEvents.
if "EventSource" in df.columns:
    slide_mask = slide_mask | df["EventSource"].astype(str).str.strip().eq("SlideEvents")

# Important sample-compatible logic: in this file EventSource is numeric, but SlideEvent is populated.
if REMOVE_SLIDE_EVENT_ROWS_STRONG and "SlideEvent" in df.columns:
    slide_mask = slide_mask | df["SlideEvent"].notna()

removed_rows_preview = df.loc[slide_mask, [c for c in ["Row", "Timestamp", "EventSource", "SlideEvent", "StimType"] if c in df.columns]].copy()

print("Rows before removal:", rows_before)
print("Slide/media rows detected:", int(slide_mask.sum()))
print("\nDetected slide/media rows:")
display(removed_rows_preview.head(20))

df = df.loc[~slide_mask].copy()
rows_after = len(df)

print("Rows after removal:", rows_after)
print("Rows removed:", rows_before - rows_after)

if "SlideEvent" in df.columns:
    remaining_slide_rows = int(df["SlideEvent"].notna().sum())
else:
    remaining_slide_rows = 0

passed = (rows_before - rows_after) == int(slide_mask.sum()) and remaining_slide_rows == 0

details = f"Remaining non-null SlideEvent rows: {remaining_slide_rows}"
print_step_result("CELL 7 — Slide/media rows removed", passed, details)

Rows before removal: 11906
Slide/media rows detected: 4

Detected slide/media rows:


,Row,Timestamp,EventSource,SlideEvent,StimType
0,1,0.0,1.0,StartSlide,TestImage
1,2,0.0,1.0,StartMedia,TestImage
11904,11905,8400.0,1.0,EndMedia,TestImage
11905,11906,8400.0,1.0,EndSlide,TestImage


Rows after removal: 11902
Rows removed: 4

CELL 7 — Slide/media rows removed
Remaining non-null SlideEvent rows: 0
✅ CLEANING APPLIED CORRECTLY


In [17]:
# ============================================================
# CELL 8 — Add Timestamp_seconds
# This cell creates a zero-based master time column in seconds.
# ============================================================

insert_at = sum(col in df.columns for col in ["subject_id", "task", "trial"])

if ADD_TIMESTAMP_SECONDS and "Timestamp_seconds" not in df.columns:
    ts, ts_source = _make_timestamp_seconds(df)
    if ts is not None:
        df.insert(insert_at, "Timestamp_seconds", ts.astype("float64"))
    else:
        ts_source = None
else:
    ts_source = "already_exists_or_disabled"

print("Timestamp source used:", ts_source)
print("Shape after Timestamp_seconds:", df.shape)

if "Timestamp_seconds" in df.columns:
    print("Timestamp_seconds summary:")
    print(df["Timestamp_seconds"].describe())
    print("Monotonic increasing:", df["Timestamp_seconds"].dropna().is_monotonic_increasing)
    print("\nPreview:")
    display(df[["Timestamp_seconds", "Timestamp"]].head(10))

passed = ("Timestamp_seconds" in df.columns) and df["Timestamp_seconds"].dropna().is_monotonic_increasing

print_step_result("CELL 8 — Timestamp_seconds added", passed)

[INFO] Timestamp → Timestamp_seconds
       median positive Δt ≈ 0.000496050 s; rough row-rate ≈ 2015.92 Hz
       Note: this is a row-level clock for interleaved streams, not native EEG/EMG/ET fs.
Timestamp source used: Timestamp
Shape after Timestamp_seconds: (11902, 78)
Timestamp_seconds summary:
count    11902.000000
mean         4.273200
std          2.370043
min          0.000000
25%          2.307128
50%          4.231249
75%          6.326373
max          8.390497
Name: Timestamp_seconds, dtype: float64
Monotonic increasing: True

Preview:


,Timestamp_seconds,Timestamp
2,0.000000e+00,8.42220
3,1.499429e-07,8.42235
4,2.000000e-03,10.42220
5,2.000150e-03,10.42235
6,4.000000e-03,12.42220
7,4.000150e-03,12.42235
8,4.496600e-03,12.91880
9,6.000150e-03,14.42235
10,6.496600e-03,14.91880
11,8.000150e-03,16.42235



CELL 8 — Timestamp_seconds added
✅ CLEANING APPLIED CORRECTLY


In [18]:
# ============================================================
# CELL 9 — Apply unit conversions and dtype cleanup
# This cell applies gaze normalization, flag binarization, EMG centering, and EEG dtype casting.
# ============================================================

shape_before_conversion = df.shape

conversion_log = apply_unit_conversions_inplace(df, keep_dist_in_mm=KEEP_DIST_IN_MM)

print("Shape before conversion:", shape_before_conversion)
print("Shape after conversion :", df.shape)
print("\nConversion log:")
for col, msg in conversion_log:
    print(f" - {col}: {msg}")

# Verification summaries
print("\nGaze range check:")
for col in ["ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty"]:
    if col in df.columns:
        s = df[col].dropna()
        if len(s):
            print(f"{col}: min={s.min():.4f}, max={s.max():.4f}, dtype={df[col].dtype}, non-null={len(s)}")
        else:
            print(f"{col}: all NaN, dtype={df[col].dtype}")

print("\nFlag unique-value check:")
for col in ["ET_ValidityLeftEye", "ET_ValidityRightEye", "ET_Blink", "ET_Fixation", "ET_Worn"]:
    if col in df.columns:
        print(f"{col}: unique={sorted(pd.Series(df[col]).dropna().unique().tolist())}, dtype={df[col].dtype}")

print("\nEMG raw median check after robust centering:")
for col in sorted(EMG_RAW_COLS):
    if col in df.columns:
        s = df[col].dropna()
        if len(s):
            print(f"{col}: median={float(s.median()):.4f}, mean={float(s.mean()):.4f}, dtype={df[col].dtype}, non-null={len(s)}")
        else:
            print(f"{col}: all NaN")

print("\nEEG dtype/non-null check:")
for ch in ["Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8"]:
    if ch in df.columns:
        print(f"{ch}: dtype={df[ch].dtype}, non-null={df[ch].notna().sum()}")

issues = []

# Gaze should be within [0,1].
for col in ["ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty"]:
    if col in df.columns:
        s = df[col].dropna()
        if len(s) and not s.between(0, 1).all():
            issues.append(f"{col} is outside [0,1].")

# Flags should be binary.
for col in ["ET_ValidityLeftEye", "ET_ValidityRightEye", "ET_Blink", "ET_Fixation", "ET_Worn"]:
    if col in df.columns:
        vals = set(pd.Series(df[col]).dropna().unique().tolist())
        if not vals.issubset({0, 1}):
            issues.append(f"{col} is not binary: {vals}")

# EMG raw should be centered if ADC-like.
for col in sorted(EMG_RAW_COLS):
    if KEEP_EMG_RAW and col in df.columns:
        s = df[col].dropna()
        if len(s) and abs(float(s.median())) > 1e-3:
            issues.append(f"{col} median is not near zero after centering.")

passed = len(issues) == 0
if issues:
    print("\nIssues found:")
    for issue in issues:
        print(" -", issue)

print_step_result("CELL 9 — Unit conversions and dtypes applied", passed)

Shape before conversion: (11902, 78)
Shape after conversion : (11902, 78)

Conversion log:
 - ET_GazeLeftx: px→[0,1] by 1600.0; clipped; dtype=float32
 - ET_GazeRightx: px→[0,1] by 1600.0; clipped; dtype=float32
 - ET_GazeLefty: px→[0,1] by 1200.0; clipped; dtype=float32
 - ET_GazeRighty: px→[0,1] by 1200.0; clipped; dtype=float32
 - ET_DistanceLeft: kept in mm; dtype=float32
 - ET_DistanceRight: kept in mm; dtype=float32
 - ET_PupilLeft: kept in mm; dtype=float32
 - ET_PupilRight: kept in mm; dtype=float32
 - ET_ValidityLeftEye: thresholded >0.5 → {0,1}; dtype=uint8
 - ET_ValidityRightEye: thresholded >0.5 → {0,1}; dtype=uint8
 - ET_Blink: thresholded >0.5 → {0,1}; dtype=uint8
 - ET_Fixation: thresholded >0.5 → {0,1}; dtype=uint8
 - ET_Worn: thresholded >0.5 → {0,1}; dtype=uint8
 - ET_GyroX: dtype=float32
 - ET_GyroY: dtype=float32
 - ET_GyroZ: dtype=float32
 - ET_AccX: dtype=float32
 - ET_AccY: dtype=float32
 - ET_AccZ: dtype=float32
 - ET_HeadRotationPitch: dtype=float32
 - ET_HeadR

In [19]:
# ============================================================
# CELL 10 — Drop metadata/noisy columns
# This cell removes vendor EMG mV, event columns, camera XY, and eyelid features.
# ============================================================

cols_before_drop = list(df.columns)

to_drop = set(DROP_COLS_EXACT_BASE)

if not KEEP_EMG_RAW:
    to_drop.update(EMG_RAW_COLS)

if DROP_EMG_VENDOR:
    to_drop.update(EMG_VENDOR_COLS)

if DROP_EYELID_FEATURES:
    to_drop.update({
        "ET_Gaze3dEyelidAngleTopLeft", "ET_Gaze3dEyelidAngleBottomLeft",
        "ET_Gaze3dEyelidAngleTopRight", "ET_Gaze3dEyelidAngleBottomRight",
        "ET_Gaze3dEyelidApertureLeft", "ET_Gaze3dEyelidApertureRight",
    })

# Drop duplicate EventSource columns produced by pandas: EventSource.1, EventSource.2, EventSource.3, etc.
to_drop.update(c for c in df.columns if c in DROP_COLS_MAYBE)
to_drop.update(c for c in df.columns if re.fullmatch(r"EventSource\.\d+", c))

# Drop unnamed columns
to_drop.update(c for c in df.columns if any(c.startswith(p) for p in DROP_PREFIXES))

drop_now = sorted([c for c in to_drop if c in df.columns])

df = df.drop(columns=drop_now, errors="ignore")

cols_after_drop = list(df.columns)

print("Columns before drop:", len(cols_before_drop))
print("Columns after drop :", len(cols_after_drop))
print("Dropped columns:")
for col in drop_now:
    print(" -", col)

print("\nRemaining columns:")
print(cols_after_drop)

bad_remaining = []
bad_remaining += [c for c in EMG_VENDOR_COLS if DROP_EMG_VENDOR and c in df.columns]
bad_remaining += [c for c in df.columns if c.startswith("EventSource")]
bad_remaining += [c for c in ["SlideEvent", "StimType", "Duration", "CollectionPhase", "SourceStimuliName"] if c in df.columns]
bad_remaining += [c for c in ["ET_CameraLeftX", "ET_CameraLeftY", "ET_CameraRightX", "ET_CameraRightY"] if c in df.columns]

passed = len(bad_remaining) == 0

if bad_remaining:
    print("\nColumns that should not remain:")
    print(bad_remaining)

print_step_result("CELL 10 — Unwanted columns dropped", passed)

Columns before drop: 78
Columns after drop : 55
Dropped columns:
 - Ch1 EMG
 - Ch2 EMG
 - Ch3 EMG
 - Ch4 EMG
 - CollectionPhase
 - Duration
 - ET_CameraLeftX
 - ET_CameraLeftY
 - ET_CameraRightX
 - ET_CameraRightY
 - ET_Gaze3dEyelidAngleBottomLeft
 - ET_Gaze3dEyelidAngleBottomRight
 - ET_Gaze3dEyelidAngleTopLeft
 - ET_Gaze3dEyelidAngleTopRight
 - ET_Gaze3dEyelidApertureLeft
 - ET_Gaze3dEyelidApertureRight
 - EventSource
 - EventSource.1
 - EventSource.2
 - EventSource.3
 - SlideEvent
 - SourceStimuliName
 - StimType

Remaining columns:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Row', 'Timestamp', 'SampleNumber', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_TimeSignal', 'ET_PupilLeft', 'ET_PupilRight', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotation

In [20]:
# ============================================================
# CELL 11 — Compact metadata/index dtypes
# This cell casts subject/task/trial/SampleNumber/Row to int32.
# ============================================================

for col in ["subject_id", "task", "trial", "SampleNumber", "Row"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int32")

print("Final shape before save:", df.shape)
print("\nImportant column dtypes:")
important_cols = [c for c in ["subject_id", "task", "trial", "Row", "SampleNumber", "Timestamp_seconds", "Timestamp"] if c in df.columns]
important_cols += [c for c in sorted(EMG_RAW_COLS) if c in df.columns]
important_cols += [c for c in ["Ch1", "Ch2", "Ch3", "Ch4", "ET_GazeLeftx", "ET_GazeLefty"] if c in df.columns]

print(df[important_cols].dtypes)

print("\nFinal preview:")
display(df.head(10))

required_meta = ["subject_id", "task", "trial", "Timestamp_seconds"]
missing_meta = [c for c in required_meta if c not in df.columns]

passed = len(missing_meta) == 0

if missing_meta:
    print("Missing required metadata/time columns:", missing_meta)

print_step_result("CELL 11 — Metadata/index dtypes compacted", passed)

Final shape before save: (11902, 55)

Important column dtypes:
subject_id             int32
task                   int32
trial                  int32
Row                    int32
SampleNumber           int32
Timestamp_seconds    float64
Timestamp            float64
Ch1 EMG raw          float32
Ch2 EMG raw          float32
Ch3 EMG raw          float32
Ch4 EMG raw          float32
Ch1                  float32
Ch2                  float32
Ch3                  float32
Ch4                  float32
ET_GazeLeftx         float32
ET_GazeLefty         float32
dtype: object

Final preview:


,subject_id,task,trial,Timestamp_seconds,Row,Timestamp,SampleNumber,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Fixation,ET_Worn,LSL Timestamp,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8
2,10,2,4,0.000000e+00,3,8.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676423,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
3,10,2,4,1.499429e-07,4,8.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676424,104424.921875,97475.304688,104743.679688,117912.734375,134634.218750,61569.078125,62151.535156,45519.785156
4,10,2,4,2.000000e-03,5,10.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678423,104432.734375,97476.398438,104738.296875,117938.625000,134648.578125,61575.230469,62170.226562,45518.308594
5,10,2,4,2.000150e-03,6,10.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678424,104432.734375,97476.398438,104738.296875,117938.625000,134648.578125,61575.230469,62170.226562,45518.308594
6,10,2,4,4.000000e-03,7,12.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.680423,104458.960938,97479.929688,104761.281250,117996.179688,134697.359375,61586.433594,62254.437500,45564.179688
7,10,2,4,4.000150e-03,8,12.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.680424,104458.960938,97479.929688,104761.281250,117996.179688,134697.359375,61586.433594,62254.437500,45564.179688
8,10,2,4,4.496600e-03,9,12.91880,393172,225.5,16.0,-169.0,-40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,2,4,6.000150e-03,10,14.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.682423,104482.617188,97487.796875,104790.367188,118037.140625,134738.796875,61601.933594,62328.203125,45617.253906
10,10,2,4,6.496600e-03,11,14.91880,393173,255.5,-72.0,-43.0,-64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,10,2,4,8.000150e-03,12,16.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.684423,104488.578125,97489.750000,104799.906250,118038.046875,134746.093750,61608.035156,62350.230469,45646.960938



CELL 11 — Metadata/index dtypes compacted
✅ CLEANING APPLIED CORRECTLY


In [21]:
# ============================================================
# CELL 12 — Final validation before saving
# This cell checks whether Phase 1A cleaning was applied correctly.
# ============================================================

issues = []

# Required columns
for col in ["subject_id", "task", "trial", "Timestamp_seconds"]:
    if col not in df.columns:
        issues.append(f"Missing required column: {col}")

# Timestamp
if "Timestamp_seconds" in df.columns:
    if not df["Timestamp_seconds"].dropna().is_monotonic_increasing:
        issues.append("Timestamp_seconds is not monotonic increasing.")
    if df["Timestamp_seconds"].dropna().min() < -1e-9:
        issues.append("Timestamp_seconds has negative values.")

# Vendor EMG should be removed
for col in EMG_VENDOR_COLS:
    if DROP_EMG_VENDOR and col in df.columns:
        issues.append(f"Vendor EMG column still present: {col}")

# Raw EMG should be kept
for col in EMG_RAW_COLS:
    if KEEP_EMG_RAW and col not in df.columns:
        issues.append(f"Raw EMG column missing: {col}")

# Gaze should be normalized
for col in ["ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty"]:
    if col in df.columns:
        s = df[col].dropna()
        if len(s) and not s.between(0, 1).all():
            issues.append(f"{col} is outside [0,1].")

# Binary ET flags
for col in ["ET_ValidityLeftEye", "ET_ValidityRightEye", "ET_Blink", "ET_Fixation", "ET_Worn"]:
    if col in df.columns:
        vals = set(pd.Series(df[col]).dropna().unique().tolist())
        if not vals.issubset({0, 1}):
            issues.append(f"{col} has non-binary values: {vals}")

# Event/noisy columns should be removed
for col in df.columns:
    if col.startswith("EventSource"):
        issues.append(f"EventSource column still present: {col}")

for col in ["SlideEvent", "StimType", "Duration", "CollectionPhase", "SourceStimuliName",
            "ET_CameraLeftX", "ET_CameraLeftY", "ET_CameraRightX", "ET_CameraRightY"]:
    if col in df.columns:
        issues.append(f"Noisy/meta column still present: {col}")

# Empty sensor rows check
sensor_cols = [c for c in [f"Ch{i}" for i in range(1, 9)] + sorted(EMG_RAW_COLS) + ["ET_TimeSignal"] if c in df.columns]
if sensor_cols:
    empty_sensor_rows = int(df[sensor_cols].isna().all(axis=1).sum())
    if empty_sensor_rows > 0:
        issues.append(f"{empty_sensor_rows} rows have no EEG/EMG/ET sensor values.")
else:
    issues.append("No EEG/EMG/ET sensor columns found.")

print("Validation checks completed.")
print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Sensor columns checked:", sensor_cols)

if issues:
    print("\nIssues found:")
    for i, issue in enumerate(issues, 1):
        print(f"{i}. {issue}")
else:
    print("\nNo validation issues found.")

passed = len(issues) == 0
print_step_result("CELL 12 — Final Phase 1A validation", passed)

Validation checks completed.
Final rows: 11902
Final columns: 55
Sensor columns checked: ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_TimeSignal']

No validation issues found.

CELL 12 — Final Phase 1A validation
✅ CLEANING APPLIED CORRECTLY


In [22]:
# ============================================================
# CELL 13 — Save cleaned CSV and verify saved file
# This cell saves the cleaned CSV and reloads it to confirm the file is correct.
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / INPUT_CSV.name

df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved cleaned CSV to:")
print(out_path)

print("\nFile exists:", out_path.exists())
print("File size MB:", round(out_path.stat().st_size / (1024 * 1024), 3))

df_saved = pd.read_csv(out_path)
print("\nReloaded saved shape:", df_saved.shape)
print("Original cleaned shape:", df.shape)

display(df_saved.head(5))

passed = out_path.exists() and (df_saved.shape == df.shape)

if passed:
    print("\n✅ CLEANING APPLIED CORRECTLY: Cleaned file saved and reloaded successfully.")
else:
    print("\n❌ CLEANING NOT FULLY APPLIED: Saved file shape does not match cleaned dataframe.")

Saved cleaned CSV to:
/content/phase1A_cleaned/063_T24.csv

File exists: True
File size MB: 2.022

Reloaded saved shape: (11902, 55)
Original cleaned shape: (11902, 55)


,subject_id,task,trial,Timestamp_seconds,Row,Timestamp,SampleNumber,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Fixation,ET_Worn,LSL Timestamp,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8
0,10,2,4,0.000000e+00,3,8.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676423,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785
1,10,2,4,1.499429e-07,4,8.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676424,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785
2,10,2,4,2.000000e-03,5,10.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678423,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310
3,10,2,4,2.000150e-03,6,10.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678424,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310
4,10,2,4,4.000000e-03,7,12.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.680423,104458.960,97479.930,104761.28,117996.180,134697.36,61586.434,62254.438,45564.180



✅ CLEANING APPLIED CORRECTLY: Cleaned file saved and reloaded successfully.


In [23]:
# ============================================================
# CELL 14 — Download cleaned CSV from Colab
# This cell lets you download the cleaned file to share with your professor.
# ============================================================

print("Cleaned file path:", out_path)

if IN_COLAB:
    print("Starting download...")
    files.download(str(out_path))
else:
    print("Not running in Colab. Download manually from:", out_path)

Cleaned file path: /content/phase1A_cleaned/063_T24.csv
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes from this debug version

- The core Phase 1A logic is preserved: metadata insertion, timestamp creation, gaze normalization, ET flag binarization, EMG raw median-centering, EEG float casting, vendor EMG removal, and noisy metadata column dropping.
- One important safety correction was added: this sample file has slide/media rows where `SlideEvent` is populated but `EventSource` is numeric (`1.0`), so the original strict `EventSource == "SlideEvents"` condition would not remove them. This notebook removes those rows using `SlideEvent.notna()` while preserving the original intention.
- The final validation cell is designed to clearly print **✅ CLEANING APPLIED CORRECTLY** only when all checks pass.